# Milestone 3 on Kaggle: LFCC + LCNN, and the GMM on In-the-Wild

Settings: **Accelerator = GPU (T4 or P100)**, **Internet = on**. Run with *Save Version, Save & Run All*.

1. downloads ASVspoof 2019 LA (ODC-By, doi:10.7488/ds/2555) and In-the-Wild (Muller et al. 2022, Hugging Face `mueller91/In-The-Wild`, CC-BY-SA 4.0);
2. extracts LFCC frames for train, dev, eval and In-the-Wild;
3. scores In-the-Wild with the milestone 1 GMMs;
4. trains the LCNN with 3 seeds and scores eval + In-the-Wild.

The wav2vec 2.0 features of In-the-Wild were extracted by the first run of this notebook (`scripts/extract_wav2vec2.py --partitions itw`), so that step is not repeated.

Every step goes through `run()`, which **stops the notebook at the first error**. The first version used `!python ...`, which does not stop on failure: a refused download then made every later cell fail for nothing.

Output, in `/kaggle/working/results`: score files and training histories (a few MB).

In [ ]:
import os, shutil, subprocess, torch, torchaudio
print('torch', torch.__version__, '| torchaudio', torchaudio.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - enable the GPU accelerator')
print('CPU cores:', os.cpu_count(), '| free disk in /tmp: %.0f GB' % (shutil.disk_usage('/tmp').free / 1e9))

def run(cmd):
    """Run a shell command, stream its output, raise if it fails."""
    print('$', cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True)

In [ ]:
run('rm -rf /tmp/repo && git clone --depth 1 https://github.com/amine-belkacem/audio-deepfake-generalization.git /tmp/repo')
run('pip install -q soundfile')
os.chdir('/tmp/repo')

In [ ]:
run('python scripts/download_asvspoof2019.py --out /tmp/data')

In [ ]:
run('python scripts/download_in_the_wild.py --out /tmp/itw')

In [ ]:
run('python scripts/extract_lfcc.py --la-root /tmp/data/LA --itw-root /tmp/itw --feat-root /tmp/features --partitions train dev eval itw')

In [ ]:
run('python scripts/score_gmm.py --gmm-dir results/lfcc_gmm_asvspoof2019_b02 --feat-dir /tmp/features/lfcc_asvspoof2019_b02/itw '
    '--out /kaggle/working/results/lfcc_gmm_asvspoof2019_b02/scores_itw.csv')

In [ ]:
run('python scripts/train_lcnn.py --feat-dir /tmp/features/lfcc_asvspoof2019_b02 --la-root /tmp/data/LA --itw-root /tmp/itw '
    '--out /kaggle/working/results/lfcc_lcnn --device cuda')

In [ ]:
run('rm -f /kaggle/working/results/lfcc_lcnn/*.pt /kaggle/working/results/lfcc_lcnn/*.npz')  # weights are not needed on the laptop
run('ls -laR /kaggle/working/results')